## 1、了解什么是提示词模板：
模板+变量值=完整的提示词，避免将提示词写死，更加灵活，好维护，适用于大模型应用程序

## 2、什么是(Langchain当中的)PromptTemplate：
一个单一文本（非消息列表的）提示词模板

In [6]:
from langchain_core.prompts import PromptTemplate
# 1、PromptTemplate的实例化：一个是直接实例化，另外一种方式：调用类方法，from_template
template = PromptTemplate(
    template="你是一个翻译助手，帮助用户将{content}翻译成语言：{lang}",
    input_variables=["content","lang"]
)
template

PromptTemplate(input_variables=['content', 'lang'], input_types={}, partial_variables={}, template='你是一个翻译助手，帮助用户将{content}翻译成语言：{lang}')

In [3]:
res =  template.format(content = "什么是LangChain",lang="英语")
print(res)

你是一个翻译助手，帮助用户将什么是LangChain翻译成语言：英语


In [5]:
# 实例化方式二
template2 = PromptTemplate.from_template(
    template="你是一个翻译助手，帮助用户将{content}翻译成语言：{lang}"
)
template2

PromptTemplate(input_variables=['content', 'lang'], input_types={}, partial_variables={}, template='你是一个翻译助手，帮助用户将{content}翻译成语言：{lang}')

In [7]:
## 2、转换成具体的prompt
# a.通过format方式去调用
res =  template.format(content = "什么是LangChain",lang="英语")
print(type(res))

<class 'str'>


In [8]:
## b. invoke方式去调用
invoke_res = template.invoke({"content":"什么是LangChain","lang":"英语"})
invoke_res.to_string()

'你是一个翻译助手，帮助用户将什么是LangChain翻译成语言：英语'

## 3、什么是ChatPromtTemplate
消息列表提示词模板，适用于多轮对话

In [10]:
from langchain_core.prompts import ChatPromptTemplate
# 1、实例化
chat_promt_template = ChatPromptTemplate(
    [
        {"role":"system","content":"你是一个{ai_role}，可以帮助用户解决问题"},
        {"role":"human","content":"{user_input}"}
    ]
)

In [15]:
type(res.to_messages())

list

In [16]:
res.to_messages()

[SystemMessage(content='你是一个翻译助手，可以帮助用户解决问题', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='将 什么是langchain 翻译成英文', additional_kwargs={}, response_metadata={})]

In [17]:
chat_promt_template2 = ChatPromptTemplate.from_messages(
    [
        {"role":"system","content":"你是一个{ai_role}，可以帮助用户解决问题"},
        {"role":"human","content":"{user_input}"}
    ]
)

In [ ]:
# 转换成消息列表
es = chat_promt_template.invoke({"ai_role":"翻译助手","user_input":"将 什么是langchain 翻译成英文"})
es

In [18]:
chat_promt_template2

ChatPromptTemplate(input_variables=['ai_role', 'user_input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['ai_role'], input_types={}, partial_variables={}, template='你是一个{ai_role}，可以帮助用户解决问题'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['user_input'], input_types={}, partial_variables={}, template='{user_input}'), additional_kwargs={})])

## 4、怎么去结合LLM去使用template

In [19]:
from langchain.chat_models import init_chat_model
import dotenv
dotenv.load_dotenv()
model = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    temperature = 0.5,
)
# 方式一：
# 1、先通过template得到具体的prompt
# 2、将prompt传递给模型的invoke方法去做调用
model.invoke(chat_promt_template.invoke({"ai_role":"翻译助手","user_input":"将 什么是langchain 翻译成英文"}))

AIMessage(content='What is LangChain?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 21, 'total_tokens': 26, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 21}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': '49e31d4c-a5ea-4f8c-88ef-47c578dc2ed0', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--f65f6a2e-d1a5-4bcd-b0b6-6928ebfffbec-0', usage_metadata={'input_tokens': 21, 'output_tokens': 5, 'total_tokens': 26, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

In [20]:
model.invoke(template2.invoke({"content":"什么是LangChain","lang":"英语"}).to_string())

AIMessage(content='What is LangChain?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 20, 'total_tokens': 25, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 20}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': '5554d876-93db-43f7-9ace-812377c8cfb1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--032cb28b-c6b4-4180-bc1e-8fbfa81bb624-0', usage_metadata={'input_tokens': 20, 'output_tokens': 5, 'total_tokens': 25, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

In [21]:
# 方式二：通过chain去调用，会在第三章 chain，讲解具体的原理
chain = template2 | model
chain.invoke({"content":"什么是LangChain","lang":"英语"})

AIMessage(content='What is LangChain?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 20, 'total_tokens': 25, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 20}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': '91755f44-50fa-407f-a1e4-106fb6d37a52', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--167d1ef8-491c-4e3e-b248-63a58dc5be7b-0', usage_metadata={'input_tokens': 20, 'output_tokens': 5, 'total_tokens': 25, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

## 5、template的一些高级特性

### 部分提示词模板
作用：适用于提示词模板的层级管理

In [14]:
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
template = PromptTemplate(
    template="你是一个{ai_role}，帮助用户解决相关问题，用户输入:{user_input}",
    input_variables=["ai_role","user_input"]
)
template

PromptTemplate(input_variables=['ai_role', 'user_input'], input_types={}, partial_variables={}, template='你是一个{ai_role}，帮助用户解决相关问题，用户输入:{user_input}')

In [17]:
translate_partial_template = template.partial(ai_role = "翻译专家")
math_partial_template =template.partial(ai_role= "数学家")
print(math_partial_template)
print(translate_partial_template)

input_variables=['user_input'] input_types={} partial_variables={'ai_role': '数学家'} template='你是一个{ai_role}，帮助用户解决相关问题，用户输入:{user_input}'
input_variables=['user_input'] input_types={} partial_variables={'ai_role': '翻译专家'} template='你是一个{ai_role}，帮助用户解决相关问题，用户输入:{user_input}'


In [19]:
math_partial_template.invoke({"user_input":"什么是微积分"})
translate_partial_template.invoke({"user_input":"翻译什么是langchain"})

StringPromptValue(text='你是一个翻译专家，帮助用户解决相关问题，用户输入:翻译什么是langchain')

### 消息占位符
作用：一般用于Agent当中维护历史对话消息列表

In [20]:
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system","你是一个{ai_role}"),
        ("placeholder","{conversation}"), # 消息占位符
    ]
)

In [23]:
res = chat_template.format(
    ai_role = "智能客服",
    conversation = [
        ("human","你是谁"),
        ("ai","我是一个智能客服")
    ]
)

In [24]:
res

'System: 你是一个智能客服\nHuman: 你是谁\nAI: 我是一个智能客服'

## 6、从文件当中加载提示词模板
可以从json或者是Yaml当中去进行加载，langchain内部会自动推导文件类型，按照相应的类型去做加载

In [38]:
from langchain_core.prompts import load_prompt
file_template = load_prompt(path=r"D:\PycharmProjects\lessons\0716_llm\langchain_demo\chapter_01\prompt.json", encoding="utf-8")

In [39]:
file_template

PromptTemplate(input_variables=['ai_role', 'user_input'], input_types={}, partial_variables={}, template='你是{ai_role}，现在用户输入{user_input}')

In [37]:
yaml_file_template = load_prompt(path=r"D:\PycharmProjects\lessons\0716_llm\langchain_demo\chapter_01\prompt2.yaml",encoding="utf-8")

ScannerError: mapping values are not allowed here
  in "D:\PycharmProjects\lessons\0716_llm\langchain_demo\chapter_01\prompt2.yaml", line 2, column 9

In [30]:
yaml_file_template

PromptTemplate(input_variables=['ai_role', 'user_input'], input_types={}, partial_variables={}, template='你是{ai_role}，现在用户输入{user_input}')

## 7、多模态提示词
通过多模态模型，可以访问图片信息，根据图片信息给到用户回应

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import dotenv
dotenv.load_dotenv()
openai_client = ChatOpenAI(
    model="gpt-4o-mini"
)
template = ChatPromptTemplate(
    [
        {"role": "system", "content": "用中文简短描述图片内容"},
        {"role": "user", "content": [{"image_url": "{image_url}"}]},
    ]
)



prompt = template.format_messages(
    image_url="https://img2.baidu.com/it/u=2976763563,2523722948&fm=253&app=138&f=JPEG?w=800&h=1200"
)
prompt

[SystemMessage(content='用中文简短描述图片内容', additional_kwargs={}, response_metadata={}),
 HumanMessage(content=[{'type': 'image_url', 'image_url': {'url': 'https://img2.baidu.com/it/u=2976763563,2523722948&fm=253&app=138&f=JPEG?w=800&h=1200'}}], additional_kwargs={}, response_metadata={})]

In [4]:
openai_client.invoke(prompt)

AIMessage(content='这是一只黑白相间的狗，微笑着朝前看，耳朵直立，表情友好。背景为淡灰色，整体给人一种温暖和愉悦的感觉。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 36853, 'total_tokens': 36900, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CZ5PcXIDV3SqH4rQn8eP0owMxAa8M', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--8e8e8193-a3c0-412f-8c4d-64f9458bfd73-0', usage_metadata={'input_tokens': 36853, 'output_tokens': 47, 'total_tokens': 36900, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
image_path = r"D:\个人\川西照片\DSC01464.jpg"
# Function to encode the image
import base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

base64_image = encode_image(image_path)

In [6]:
from langchain_openai import ChatOpenAI
import dotenv
dotenv.load_dotenv()
client = ChatOpenAI(
    model="gpt-4o-mini"
)

In [9]:
input=[
    {
      "role":"system","content":"根据用户输入图片，对图片内容进行描述"
    },
        {
            "role": "user",
            "content": [
                # { "type": "input_text", "text": "what's in this image?" },
                {
                    # "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }
    ]
client.invoke(input)

AIMessage(content='The image depicts two young individuals walking along a road, holding bicycles. They are set against a scenic backdrop of green mountains and a blue sky. The person on the left wears a brown leather jacket and khaki pants, while the other individual on the right is dressed in a fringed brown jacket over a plaid skirt. Both appear relaxed and are smiling, enjoying the moment together. The surroundings suggest a natural and picturesque landscape, typical of rural areas.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 36855, 'total_tokens': 36945, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-CZ8bbdfcHprkAhwymRFJqHrd6p8qC', '